# DDRI bike_change full161 HMW 정렬 회귀앙상블 Run-All 노트북

이 노트북은 서비스 대상 `161개` 대여소 기준으로, HMW 원본 flow 피처 체계를 유지한 `bike_change` 회귀앙상블 실험을 한 번에 실행합니다.

## 노트북에서 하는 일
1. 실험 스크립트와 입력 파일 경로를 확인합니다.
2. `161개` lookup과 원본 flow 파일 존재 여부를 확인합니다.
3. 본 실험 스크립트를 실행해 `train_2023 / validation_2024 / test_2025_refit` 결과를 생성합니다.
4. 모델 비교표와 최적 모델 요약을 바로 확인합니다.


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
import time

import pandas as pd
from IPython.display import Markdown, display

root_candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
script_name = "10_run_ddri_bike_change_full161_hmw_aligned_regression_ensemble.py"

script_path = None
for base in root_candidates:
    candidate = (base / "works" / "07_prediction_bike_change" / script_name).resolve()
    if candidate.exists():
        script_path = candidate
        break
    candidate = (base / script_name).resolve()
    if candidate.exists():
        script_path = candidate
        break

if script_path is None:
    found = list(Path.cwd().rglob(script_name))
    if found:
        script_path = found[0].resolve()

if script_path is None:
    raise FileNotFoundError("실험 스크립트를 찾지 못했습니다.")

work_dir = script_path.parent
root_dir = work_dir.parents[1]
data_dir_candidates = []
if os.environ.get("DDRI_HMW_DATA_DIR"):
    data_dir_candidates.append(Path(os.environ["DDRI_HMW_DATA_DIR"]).expanduser())
data_dir_candidates.append(root_dir / "3조 공유폴더" / "station_hour_bike_flow_2023_2025")
data_dir_candidates.append(root_dir / "hmw" / "Data")

train_path = None
test_path = None
data_dir = None
for candidate_dir in data_dir_candidates:
    candidate_train = candidate_dir / "station_hour_bike_flow_train_2023_2024.csv"
    candidate_test = candidate_dir / "station_hour_bike_flow_test_2025.csv"
    if candidate_train.exists() and candidate_test.exists():
        data_dir = candidate_dir
        train_path = candidate_train
        test_path = candidate_test
        break

lookup_path = root_dir / "cheng80" / "api_output" / "ddri_station_id_api_lookup.csv"

print(f"스크립트 경로: {script_path}")
print(f"작업 폴더: {work_dir}")
print(f"데이터 폴더: {data_dir}")
print(f"train 파일: {train_path}")
print(f"test 파일: {test_path}")
print(f"lookup 파일: {lookup_path}")

if train_path is None or test_path is None or not lookup_path.exists():
    raise FileNotFoundError("필수 입력 파일(flow 또는 lookup)이 없습니다.")


In [ ]:
train_head = pd.read_csv(train_path, nrows=5)
lookup_df = pd.read_csv(lookup_path)

display(Markdown(f"### flow train 샘플 ({train_path.name})"))
display(train_head)
display(Markdown(f"### 161개 lookup 개수: `{len(lookup_df)}`"))


In [ ]:
env = os.environ.copy()
env.setdefault("PYTHONUNBUFFERED", "1")

t0 = time.perf_counter()
process = subprocess.Popen(
    [sys.executable, "-u", str(script_path)],
    cwd=str(root_dir),
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in process.stdout:
    print(line, end="")

return_code = process.wait()
elapsed = time.perf_counter() - t0
print(f"총 실행시간: {elapsed:,.2f}초")
if return_code != 0:
    raise RuntimeError(f"실험 스크립트 실행 실패: return_code={return_code}")


In [ ]:
metrics_path = work_dir / "output" / "data" / "ddri_bike_change_full161_hmw_aligned_model_metrics.csv"
meta_path = work_dir / "ddri_bike_change_full161_hmw_aligned_meta.json"
summary_path = work_dir / "ddri_bike_change_full161_hmw_aligned_summary.md"

metrics_df = pd.read_csv(metrics_path)
display(Markdown("### 모델 비교표"))
display(metrics_df.sort_values(["split", "RMSE"]).reset_index(drop=True))

meta = json.loads(meta_path.read_text(encoding="utf-8"))
display(Markdown(f"**최적 모델**: `{meta['best_model']}`  \n- **최적 test RMSE**: `{meta['best_test_rmse']:.6f}`  \n- **elapsed_seconds**: `{meta['elapsed_seconds']:.2f}`"))
display(Markdown(summary_path.read_text(encoding="utf-8")))
